# Mitate benchmark on Kaggle

Runs `bench.run` on a Kaggle GPU (research ledger D21) for one dataset per run: stage 1 (every
baseline, per store size) or, once every size is finished, stage 2 (Mitate, whose ranker is
trained on the validation data of all stores together).

* `DATASET = "otto"`: OTTO stores of 30K events and up. Smaller OTTO stores ran locally, and a
  fingerprint check proves this machine samples the identical stores.
* `DATASET = "hm"`: H&M fashion at every store size. The competition data never leaves Kaggle:
  the saved output holds scores and recommendation lists in store-local ids, nothing else.

Every model in a store-size bucket must run on the same accelerator, because the 900 s neural
trial cap binds at large sizes and a faster GPU buys more epochs.

**Before running (Settings panel, right side):**

1. *Accelerator*: **GPU T4 x2**. Keep the same choice for every session of a bucket.
2. *Internet*: **On** (installs the pinned packages; fetches the pinned GRU4Rec code).
3. *Input*: add your private dataset **mitate-bench**. For H&M, also add the competition
   **H&M Personalized Fashion Recommendations** (accept its rules first). When resuming, also
   add the output of this notebook's previous version (*Add Input → Your Work → Notebook Output
   Files*).
4. Set `DATASET`, `BUCKETS` and `STAGE` below, then **Save Version → Save & Run All (Commit)**. A commit
   keeps running with the browser closed, for up to 12 hours.

Each run saves everything finished so far in `mitate-results/` (download it from the version's
*Output* tab). Finished (store, model) results are skipped on resume, so an interrupted bucket
continues where it stopped.

In [ ]:
# ---- settings -------------------------------------------------------------------------------
DATASET = "otto"  # "otto" (stores of 30K events and up) or "hm" (every size)
BUCKETS = [30000]  # one bucket per run is easiest, smallest first
STAGE = "1"  # "1": baselines on BUCKETS; "2": Mitate, trained on every finished store
RUN_HOURS = 11.0  # stop starting new work after this, leaving time to save outputs
# The local store otto-n1000-s1 hashes to this; Kaggle must sample the identical store.
EXPECTED_FINGERPRINT = "d5376154c1bb020cd8884ea8db67a72d2cd5515fbe454de2e26348d6e7bee4b2"
RESUME = 0  # set by kernel.sh: finished results the attached results dataset must hold
CODE_VERSION = None  # set by scripts/kaggle/kernel.sh: the run stops if the mounted code differs

In [ ]:
import fnmatch
import json
import os
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

T0 = time.time()
INPUT, WORK, REPO = Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp/mitate")
RESULTS = REPO / "experiments" / "results" / DATASET
LOG = REPO / "experiments" / "results" / f"run_kaggle_{DATASET}.log"
# one walk, following symlinked mounts (rglob does not); the H&M input holds 105K product images
INDEX = sorted(
    Path(root) / name
    for root, dirs, files in os.walk(INPUT, followlinks=True)
    for name in dirs + files
)
print("inputs:", *sorted({"/".join(p.relative_to(INPUT).parts[:3]) for p in INDEX}), sep="\n  ")


def find(pattern, required=True):
    hit = next((p for p in INDEX if fnmatch.fnmatch(p.name, pattern)), None)
    if hit is None and required:
        raise FileNotFoundError(f"No {pattern} under {INPUT}: add the input that provides it.")
    return hit


def sh(cmd, **kw):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=kw.pop("cwd", REPO), **kw)


# code: Kaggle may keep the zip or unpack it
if REPO.exists():
    shutil.rmtree(REPO)
z = find("mitate-code.zip", required=False)
if z and z.is_file():
    zipfile.ZipFile(z).extractall(REPO)
else:
    src = next(
        p.parent for p in INDEX if p.name == "pyproject.toml" and (p.parent / "bench").is_dir()
    )
    shutil.copytree(src, REPO)
mounted = (REPO / "CODE_VERSION").read_text().strip() if (REPO / "CODE_VERSION").exists() else None
if CODE_VERSION and mounted != CODE_VERSION:
    raise RuntimeError(
        f"mitate-bench holds code {mounted}, not {CODE_VERSION}. Kaggle mounts the newest dataset "
        "version only once it has finished processing: wait a few minutes, then run again."
    )
print("code", mounted, "ready in", REPO)

In [ ]:
# environment: the exact locked versions, then the CUDA build of the same torch release
sh("pip install -q uv==0.6.6", cwd=WORK)
sh("uv sync --frozen --no-dev")
PY = str(REPO / ".venv" / "bin" / "python")
torch_version = subprocess.run(
    [PY, "-c", "import torch; print(torch.__version__.split('+')[0])"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
# PyPI's Linux torch wheels are CUDA builds; the lockfile pins the CPU build for servers
sh(
    f"uv pip install --python {PY} --reinstall-package torch 'torch=={torch_version}' "
    "--index-url https://pypi.org/simple"
)
# a real kernel launch: a wheel can see the GPU yet ship no kernels for its architecture
check = subprocess.run(
    [
        PY,
        "-c",
        "import torch; ok = torch.cuda.is_available() and "
        "float((torch.ones(4, device='cuda') * 2).sum()) == 8.0; "
        "print(torch.__version__, ok, torch.cuda.get_device_name(0) if ok else '')",
    ],
    capture_output=True,
    text=True,
).stdout.strip()
print(check)
assert " True " in f" {check} ", "No CUDA GPU: set Accelerator to GPU T4 x2 and rerun."

In [ ]:
# data: OTTO links the uploaded Parquet; H&M is built from the competition files on this machine
data = REPO / "data" / DATASET
data.mkdir(parents=True, exist_ok=True)
if DATASET == "otto":
    for name in ("train.parquet", "manifest.json", "cobrowse-t*.parquet"):
        f = find(name)
        (data / f.name).symlink_to(f)
else:
    sh(f"{PY} -m bench.data.hm {find('transactions_train.csv').parent} {data}")
print(sorted(p.name for p in data.iterdir()))

In [ ]:
# resume: results of earlier runs, from a previous version's output or a results dataset. Kaggle
# unpacks an uploaded zip into the dataset's root, so a results tree is found by its contents.
for prev in INDEX:
    if prev.is_dir() and (prev / "experiments" / "results").is_dir():
        shutil.copytree(prev, REPO, dirs_exist_ok=True)
    elif prev.name == "mitate-results.zip" and prev.is_file():
        zipfile.ZipFile(prev).extractall(REPO)
    else:
        continue
    print("resumed from", prev)
done = sorted(f"{p.parent.parent.name}/{p.stem}" for p in RESULTS.glob("*/test/*.json"))
print(len(done), "finished (store, model) results already present")
if len(done) < RESUME:
    raise RuntimeError(
        f"The results dataset should hold {RESUME} finished results, not {len(done)}. Kaggle "
        "mounts a new dataset version only once it has finished processing: wait, then rerun."
    )

In [ ]:
# faithfulness: sampling a known store from scratch must reproduce the local machine exactly
if DATASET == "otto":
    stores = REPO / "data" / "stores" / "otto"
    probe = next(stores.rglob("otto-n1000-s1.npz"), None) if stores.exists() else None
    if probe:
        probe.unlink()  # never trust a copied store for this check
    out = subprocess.run(
        [PY, "-m", "bench.fingerprint", "experiments/configs/benchmark.yaml", "otto", "1000", "1"],
        cwd=REPO,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print(out)
    assert out.endswith(EXPECTED_FINGERPRINT), (
        "Store sampling differs from the local machine: stop and report this."
    )
else:
    print("H&M runs only here, so every store of it is sampled on the same machine.")

In [ ]:
# configuration: the benchmark of record, with neural training on CUDA instead of Apple MPS
import yaml

base = REPO / "experiments/configs/benchmark.yaml"
cfg = yaml.safe_load(base.read_text())
before = cfg["tuning"]["neural_device"]
cfg["tuning"]["neural_device"] = "cuda"
kaggle_cfg = REPO / "experiments/configs/benchmark.kaggle.yaml"
kaggle_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False))


def query(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


hardware = {
    "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(T0)),
    "dataset": DATASET,
    "stage": STAGE,
    "code_version": mounted,
    "buckets": BUCKETS,
    "gpu": query("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader"),
    "cpu": query("lscpu | grep 'Model name' | sed 's/.*: *//'"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
    "torch": check,
    "python": query(f"{PY} --version"),
    "config_changes": {"tuning.neural_device": [before, "cuda"]},
}
if DATASET == "otto":
    hardware["store_fingerprint_otto-n1000-s1"] = EXPECTED_FINGERPRINT
hw_dir = RESULTS / "_hardware"
hw_dir.mkdir(parents=True, exist_ok=True)
(hw_dir / f"kaggle-{time.strftime('%Y%m%d-%H%M%S', time.gmtime(T0))}.json").write_text(
    json.dumps(hardware, indent=1)
)
print(json.dumps(hardware, indent=1))

In [ ]:
# run: the chosen stage; stops starting new work at the deadline
deadline = T0 + RUN_HOURS * 3600
stopped_at_deadline = False
cmd = [
    PY,
    "-m",
    "bench.run",
    str(kaggle_cfg),
    "--datasets",
    DATASET,
    "--stage",
    STAGE,
    # stage 2 pools every finished store of the dataset, so it never narrows to some buckets
    *(["--buckets", *map(str, BUCKETS)] if STAGE == "1" else []),
]
print(" ".join(cmd), flush=True)
with open(LOG, "a") as log:
    stamp = time.strftime("%Y-%m-%d %H:%M:%S")
    log.write(f"=== kaggle run {stamp} {DATASET} stage={STAGE} buckets={BUCKETS} ===\n")
    proc = subprocess.Popen(
        cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    os.set_blocking(proc.stdout.fileno(), False)
    while proc.poll() is None:
        line = proc.stdout.readline()
        if line:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        else:
            time.sleep(2)
        if time.time() > deadline:
            print("Time budget reached: stopping. Unfinished models rerun on resume.", flush=True)
            stopped_at_deadline = True
            proc.terminate()
            proc.wait(timeout=120)
            break
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
print("exit code", proc.returncode)

In [ ]:
# save: everything needed to resume, and to merge into the local repository
keep = [RESULTS, LOG, REPO / "experiments/configs/benchmark.kaggle.yaml"]
if DATASET == "otto":
    keep.append(REPO / "data/stores/otto")  # CC BY 4.0; saves resampling on resume
# H&M stores are competition data: they stay here and are resampled, identically, on resume
dest = WORK / "mitate-results"
if dest.exists():
    shutil.rmtree(dest)
for src in keep:
    rel = src.relative_to(REPO)
    if src.is_dir():
        shutil.copytree(src, dest / rel, ignore=shutil.ignore_patterns("*.tmp"))
    elif src.exists():
        (dest / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest / rel)
status = {"exit_code": proc.returncode, "stopped_at_deadline": stopped_at_deadline}
dest.mkdir(parents=True, exist_ok=True)
(dest / "RUN_STATUS.json").write_text(json.dumps(status))
shutil.make_archive(str(WORK / "mitate-results"), "zip", dest)
for b in BUCKETS:
    for s in sorted(RESULTS.glob(f"{DATASET}-n{b}-s*")):
        print(s.name, len(list((s / "test").glob("*.json"))), "models finished")
if proc.returncode != 0 and not stopped_at_deadline:
    print(f"THE BENCHMARK FAILED (exit code {proc.returncode}): see {LOG.name} above.")
print("Download mitate-results.zip from this version's Output tab.")